In [1]:
import numpy as np
#from qiskit.primitives import Sampler
from qiskit_algorithms import AmplitudeEstimation
from qiskit_finance.circuit.library import NormalDistribution
from qiskit_finance.applications import FixedIncomePricing

In [2]:
#
# Setup your account
# You can also save your account locally using the class method QrRuntimeService.save_account(...) and
# invoke the QrRuntimeService class constructor without any arguments.
#

import os

my_token = os.environ["QR_TOKEN"]
my_name = os.environ["QR_ACCOUNT"]

#
# Set the backend of your choice, depending upon the task and your hardware configuration.
# See SDK documentation for additional help.
#

my_backend = "scarlet_quantum_rings"

In [3]:
# Create a suitable multivariate distribution
num_qubits = [2, 2]
bounds = [(0, 0.12), (0, 0.24)]
mvnd = NormalDistribution(
    num_qubits, mu=[0.12, 0.24], sigma=0.01 * np.eye(2), bounds=bounds
)

# Create fixed income component
fixed_income = FixedIncomePricing(
    num_qubits,
    np.eye(2),
    np.zeros(2),
    cash_flow=[1.0, 2.0],
    rescaling_factor=0.125,
    bounds=bounds,
    uncertainty_model=mvnd,
)

In [4]:

from quantumrings.toolkit.qiskit import QrRuntimeService
from quantumrings.toolkit.qiskit import QrSamplerV2 as Sampler

qr_services = QrRuntimeService(name = my_name, token = my_token)
qr_backend = qr_services.backend(name = my_backend, precision = "single")

 
sampler = Sampler(backend = qr_backend, run_options= {"performance":"HighestEfficiency"})

In [5]:
# the FixedIncomeExpectedValue provides us with the necessary rescalings

# create the A operator for amplitude estimation
problem = fixed_income.to_estimation_problem()

# Set number of evaluation qubits (samples)
num_eval_qubits = 5

# Construct and run amplitude estimation
#sampler = Sampler()
algo = AmplitudeEstimation(num_eval_qubits=num_eval_qubits, sampler=sampler)
result = algo.estimate(problem)

print(f"Estimated value:\t{fixed_income.interpret(result):.4f}")
print(f"Probability:    \t{result.max_probability:.4f}")

Estimated value:	2.4600
Probability:    	0.8535


In [6]:
import tutorial_magics

%qiskit_version_table
%quantumrings_version_table
%qiskit_copyright

Software,Version
QuantumRingsLib,0.11.0
quantumrings-toolkit-qiskit,0.1.20
